# CheckIt.AI — Pipeline de transformation et schéma de données

**Étape 03 — Transformer, contrôler et préparer le chargement**  
**État du projet vérifié le 2 septembre 2026**

## 1. Objectif et grain des données

Cette étape transforme les lots bruts des quatre extracteurs en un contrat commun, reproductible et journalisé. Le fichier principal chargé par Airflow dans PostgreSQL est `data/processed/publications.parquet`.

Son grain est : **une ligne par publication unique, associée à exactement une image locale validée**.

- **clé primaire logique :** `publication_id` ;
- **nombre de colonnes :** 18 ;
- **image :** obligatoire et stockée sur la même ligne que son texte ;
- **label :** facultatif, conservé sans réinterprétation ;
- **dédoublonnage secondaire :** `source_url + image_sha256`.

Le schéma conceptuel reprend volontairement cette structure plate. Il correspond à la table PostgreSQL réellement alimentée par le pipeline, sans ajouter d’entités inutiles.


## 2. Pipeline de l’acquisition à la transformation

```mermaid
flowchart TB
    subgraph SOURCES["SOURCES ET ACQUISITION"]
        direction LR
        N0["NewsData.io<br/>API REST JSON"]
        P0["PolitiFact<br/>RSS + HTML intégré"]
        F0["Fakeddit<br/>TSV + URLs d'images"]
        T0["The Conversation<br/>Atom + pages HTML"]
    end

    subgraph EXTRACT["EXTRACT — traitement adapté à la source"]
        direction LR
        N1["Requests<br/>clé, filtres, pagination"]
        P1["Requests + Feedparser<br/>contenu + vignette"]
        F1["csv.DictReader<br/>filtre texte-image"]
        T1["Feedparser + Requests<br/>+ Beautiful Soup"]
    end

    N0 --> N1
    P0 --> P1
    F0 --> F1
    T0 --> T1

    N1 --> N2[("JSON brut + image<br/>+ preuve SHA-256")]
    P1 --> P2[("JSON brut + image<br/>+ preuve SHA-256")]
    F1 --> F2[("JSON brut + image<br/>+ preuve SHA-256")]
    T1 --> T2[("JSON brut + image<br/>+ preuve SHA-256")]

    subgraph MAP["TRANSFORM — mapping vers les 18 champs"]
        direction LR
        MN["NewsData<br/>repli du contenu + langue"]
        MP["PolitiFact<br/>HTML → texte + verdict"]
        MF["Fakeddit<br/>date Unix + labels 2/3/6"]
        MT["Conversation<br/>champs HTML + langue fr"]
    end

    N2 --> MN
    P2 --> MP
    F2 --> MF
    T2 --> MT
    MN --> C1
    MP --> C1
    MF --> C1
    MT --> C1

    subgraph COMMON["TRANSFORM — traitement commun"]
        direction LR
        C1["Nettoyage et<br/>normalisation"]
        C2["Validation<br/>publication"]
        C3["Validation<br/>image et provenance"]
        C4["Dédoublonnage"]
        C5["Typage et<br/>tri stable"]
        C1 --> C2 --> C3 --> C4 --> C5
    end

    C2 -->|"invalide"| R[("invalid_records.jsonl")]
    C3 -->|"invalide"| R
    C4 -->|"doublon"| R
    C5 --> O1[("publications.parquet")]
    C5 --> O2[("publications.jsonl")]
    C5 --> O3[("manifest + logs")]
```


## 3. Traitement propre à chaque source

| Source | Acquisition | Mapping vers le contrat commun |
|---|---|---|
| NewsData.io | API REST avec `Requests`, clé d’environnement, filtres, pagination et retries | `article_id` devient `newsdata_<id>` ; le texte suit le repli `content → description → ai_summary` ; `french` devient `fr` |
| PolitiFact | RSS téléchargé avec `Requests`, puis lu avec `Feedparser` | SHA-256 de l’ID RSS ou de l’URL sur 16 caractères ; `BeautifulSoup` nettoie `content_html` et lit le verdict du `flat-meter` |
| Fakeddit | TSV lu ligne par ligne avec `csv.DictReader` ; seules les lignes texte-image exploitables sont retenues | ID Reddit préfixé par `fakeddit_` ; date Unix vers UTC ; labels 2/3/6 regroupés dans une chaîne JSON |
| The Conversation France | Flux Atom pour les URLs, puis page téléchargée et analysée avec `Beautiful Soup` | ID numérique de l’article, ou hash stable de l’URL ; texte HTML déjà extrait ; langue fixée à `fr` ; aucun label inventé |

Les quatre fonctions `transform_<source>()` produisent le même dictionnaire provisoire. Le traitement commun peut alors appliquer les mêmes validations à toutes les sources.


## 4. Traitement commun, dédoublonnage et journalisation

Le script `scripts/transform_data.py` suit quatre étapes visibles dans `logs/transform.log` :

1. **Lecture :** contrôle de chaque enveloppe JSON, de sa source, de sa date de collecte et du nombre d’enregistrements.
2. **Mapping et validation :** nettoyage Unicode et espaces, dates UTC, domaine, langue, URLs, labels et preuves d’image.
3. **Dédoublonnage :** conservation de la première ou de la dernière occurrence selon `--duplicate-policy`.
4. **Export :** typage avec pandas, tri stable, écriture atomique et création du manifeste.

Ce déroulé est découpé en fonctions ciblées : `map_and_validate_record()` traite une publication, `transform_batch()` isole un lot, `transform_inputs()` consolide les lots et `run_transformation()` pilote les sorties. Cette séparation permet de tester ou de faire évoluer une règle sans dupliquer le pipeline.

Un doublon est détecté par :

- la même valeur de `publication_id` ; ou
- le même couple `source_url + image_sha256`.

Chaque rejet contient la source, le fichier d’entrée, l’identifiant brut et la raison. Le manifeste enregistre les paramètres, les métriques, les hash des entrées et sorties, la version du schéma et le hash du pipeline.

Paramètres principaux :

| Paramètre | Rôle |
|---|---|
| `--sources` | Sélectionner une ou plusieurs sources |
| `--input-mode all/latest` | Lire tous les lots ou le dernier de chaque source |
| `--duplicate-policy keep-first/keep-latest` | Choisir l’occurrence conservée |
| `--output-format parquet/jsonl/both` | Choisir les fichiers produits |


## 5. Contrat final de 18 colonnes

| Champ | Type final | Obligatoire | Rôle |
|---|---|---:|---|
| `publication_id` | chaîne | Oui | Identifiant stable et PK logique |
| `source_name` | chaîne | Oui | Source d’acquisition |
| `source_domain` | chaîne | Oui | Domaine éditorial ou cible |
| `source_url` | URL HTTP(S) | Oui | Traçabilité de la publication |
| `title` | chaîne | Conditionnel | Texte court ; titre ou texte obligatoire |
| `text` | texte long | Conditionnel | Contenu destiné au NLP |
| `image_url` | URL HTTP(S) | Oui | URL distante de l’image liée |
| `image_path` | chemin | Oui | Image locale validée |
| `image_size_bytes` | entier | Oui | Contrôle du fichier vide ou modifié |
| `image_sha256` | chaîne hexadécimale | Oui | Intégrité et dédoublonnage |
| `image_provenance_status` | chaîne | Oui | État de la preuve URL-image |
| `published_at` | date-heure UTC | Oui | Date de publication |
| `language` | ISO 639-1 | Oui | `fr`, `en`, etc. |
| `author` | chaîne | Non | Métadonnée éditoriale |
| `source_label_raw` | chaîne | Non | Label original non converti |
| `source_label_scheme` | chaîne | Non | Système donnant son sens au label |
| `label_provenance` | chaîne | Non | Auteur ou méthode de l’annotation |
| `collected_at` | date-heure UTC | Oui | Date de collecte du lot |

### Génération de `publication_id`

| Source | Règle |
|---|---|
| NewsData.io | `newsdata_` + `article_id` fourni par l’API |
| PolitiFact | `politifact_` + 16 premiers caractères du SHA-256 de l’ID RSS ou de l’URL |
| Fakeddit | `fakeddit_` + ID Reddit fourni dans le TSV |
| The Conversation | `theconversation_` + ID numérique de l’article ; à défaut, 16 caractères du SHA-256 de l’URL |

Le préfixe évite les collisions entre deux sources qui utiliseraient le même identifiant natif.


<div class="page-break"></div>

## 6. Schéma conceptuel pour la phase Load

Le contrat final contient une seule entité. Chaque ligne regroupe une publication, son texte, son image, ses métadonnées et son éventuel label. Le manifeste de transformation conserve séparément les informations d’exécution du pipeline.

```mermaid
erDiagram
    PUBLICATION_MULTIMODALE {
        string publication_id PK
        string source_name
        string source_domain
        string source_url
        string title
        text text
        string image_url
        string image_path
        integer image_size_bytes
        string image_sha256
        string image_provenance_status
        datetime published_at
        string language
        string author
        string source_label_raw
        string source_label_scheme
        string label_provenance
        datetime collected_at
    }
```

### Lecture du schéma

`PUBLICATION_MULTIMODALE` représente une publication complète et directement exploitable :

- `publication_id` est la **PK** et identifie la ligne ;
- `title` et `text` représentent la partie textuelle ;
- les cinq champs `image_*` représentent l’image associée et sa preuve d’intégrité ;
- les champs `source_*`, `author`, `published_at`, `language` et `collected_at` assurent la traçabilité ;
- les trois champs de label restent facultatifs et indissociables.

Aucune **FK** n’est nécessaire puisqu’il n’existe qu’une table. Le lien texte-image est garanti par leur présence sur la même ligne et par les contrôles effectués pendant Transform.


## 7. Contraintes et chargement

### Contraintes d’intégrité

- `publication_id` est unique et non nul ;
- `source_name` et `source_domain` sont non nuls ;
- `title` ou `text` doit être renseigné ;
- `source_url` et `image_url` sont des URLs HTTP(S) ;
- `image_path` désigne un fichier local validé ;
- `image_size_bytes > 0` ;
- `image_sha256` contient 64 caractères hexadécimaux ;
- `image_provenance_status` indique comment l’association URL-image a été vérifiée ;
- `published_at` et `collected_at` sont des dates UTC ;
- `language` contient deux lettres minuscules ;
- les trois champs de label sont tous présents ou tous absents ;
- le couple `source_url + image_sha256` est unique après dédoublonnage.

### Chargement réalisé

Le DAG charge le Parquet dans la table `PUBLICATION_MULTIMODALE`, qui possède les mêmes 18 colonnes :

```text
publications.parquet
        │
        ├── contrôle du schéma et des contraintes
        │
        └── insertion ou mise à jour sur publication_id
                         │
                         ▼
             PUBLICATION_MULTIMODALE
```

Le manifeste reste séparé comme fichier d’audit du pipeline ; il n’est pas mélangé aux publications.


## 8. État actuel et lecture correcte des résultats

Le dossier `data/raw/` contient actuellement **21 lots et 1 707 occurrences brutes** réparties entre les quatre sources. Chaque occurrence possède une preuve complète reliant l’URL source à l’image locale.

Le dernier manifeste `data/processed/transformation_manifest.json` correspond à une exécution consolidée des quatre sources :

- 21 lots et 1 707 occurrences lus ;
- 1 077 publications uniques exportées ;
- 630 doublons écartés ;
- 0 rejet de validation ;
- schéma `1.2` et 18 colonnes.

Le Parquet actuel est donc la sortie consolidée et validée des quatre sources. Il peut être reconstruit avec :

```bash
uv run python scripts/transform_data.py --sources newsdata politifact fakeddit theconversation --input-mode all --output-format both
```
